In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

In [2]:
np.random.seed(42)
torch.manual_seed(42)

In [3]:
df = pd.read_csv('iris.csv')

df.head()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


In [4]:
list(df.columns)

['Id',
 'SepalLengthCm',
 'SepalWidthCm',
 'PetalLengthCm',
 'PetalWidthCm',
 'Species']

In [5]:
df.shape

(150, 6)

In [6]:
df.isnull().sum()

Id               0
SepalLengthCm    0
SepalWidthCm     0
PetalLengthCm    0
PetalWidthCm     0
Species          0
dtype: int64

In [7]:
df = df.drop(columns=['Id'])
df.head()

,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [8]:
species_to_number = {
    "Iris-setosa": 0,
    "Iris-versicolor": 1,
    "Iris-virginica": 2
}

encoded_targets = df['Species'].map(species_to_number)
targets_series = encoded_targets

print(encoded_targets.head())
print(encoded_targets.value_counts())

del df['Species']

print()
print(list(df.columns))

0    0
1    0
2    0
3    0
4    0
Name: Species, dtype: int64
Species
0    50
1    50
2    50
Name: count, dtype: int64

['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']


### Train-test split

In [9]:
all_inputs_numpy = df.to_numpy()
all_targets_numpy = targets_series.to_numpy()

number_of_rows = all_inputs_numpy.shape[0]

all_indices = np.arange(number_of_rows)
shuffled_indices = np.random.permutation(all_indices)

number_of_testing_rows = int(number_of_rows * 0.2)
number_of_training_rows = number_of_rows - number_of_testing_rows

training_indices = shuffled_indices[:number_of_training_rows]
testing_indices = shuffled_indices[number_of_training_rows:]

training_inputs_numpy = all_inputs_numpy[training_indices]
testing_inputs_numpy = all_inputs_numpy[testing_indices]
training_targets_numpy = all_targets_numpy[training_indices]
testing_targets_numpy = all_targets_numpy[testing_indices]

In [10]:
print('training input: ', training_inputs_numpy.shape)
print('testing input: ', testing_inputs_numpy.shape)
print('training targets: ', training_targets_numpy.shape)
print('testing target shape: ', testing_targets_numpy.shape)

training input:  (120, 4)
testing input:  (30, 4)
training targets:  (120,)
testing target shape:  (30,)


In [11]:
# + 1e-6 is for the edge case to avoid the null division
training_inputs_normalized = (training_inputs_numpy - training_inputs_numpy.mean(axis=0)) / (training_inputs_numpy.std(axis=0) + 1e-6)
testing_inputs_normalized = (testing_inputs_numpy - training_inputs_numpy.mean(axis=0)) / (training_inputs_numpy.std(axis=0) + 1e-6)

In [12]:
training_inputs_tensor = torch.tensor(training_inputs_normalized, dtype=torch.float32)
testing_inputs_tensor = torch.tensor(testing_inputs_normalized, dtype=torch.float32)

training_targets_tensor = torch.tensor(training_targets_numpy, dtype=torch.long)
testing_targets_tensor = torch.tensor(testing_targets_numpy, dtype=torch.long)

print("train input tensor: ", training_inputs_tensor.shape, training_inputs_tensor.dtype)
print("train target tensor:", training_targets_tensor.shape, training_targets_tensor.dtype)
print("test input tensor:  ", testing_inputs_tensor.shape, testing_inputs_tensor.dtype)
print("test target tensor: ", testing_targets_tensor.shape, testing_targets_tensor.dtype)

train input tensor:  torch.Size([120, 4]) torch.float32
train target tensor: torch.Size([120]) torch.int64
test input tensor:   torch.Size([30, 4]) torch.float32
test target tensor:  torch.Size([30]) torch.int64


#### 4 input feautres to hidden layer with 8 neurons to ReLU to output scores

The model returns the raw scores (logits). there is no softmax at the end sinc `CrossEntropyLoss` applies log-softmax internally by itself

ReLu keeps positive values and replaces negatives with 0. without it the two linear layers would just collapse into one big linear function and the network could not learn curved boundaries.

In [13]:
class IrisNeuralNetwork(nn.Module):

    def __init__(self):
        super().__init__()

        self.hidden_layer = nn.Linear(4, 8)
        self.relu = nn.ReLU()

        # produces 3 class scores out of 8 hidden values, one score for every species
        self.output_layer = nn.Linear(8, 3)

    def forward(self, input_data):
            hidden_output = self.hidden_layer(input_data)
            activated_hidden_output = self.relu(hidden_output)
            final_output = self.output_layer(activated_hidden_output)
            return final_output

model = IrisNeuralNetwork()
model

IrisNeuralNetwork(
  (hidden_layer): Linear(in_features=4, out_features=8, bias=True)
  (relu): ReLU()
  (output_layer): Linear(in_features=8, out_features=3, bias=True)
)

In [14]:
loss_function = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr = 0.01)

## Training loop

Since the dataset in quite small, I train on the whole training set at once in every epoch.

every epoch is going through clearing old gradients, forward passing, calculations the loss, backward passing, and at the end updating the weights.

In [15]:
for epoch_number in range(1, 201):
    model.train()

    # clearing the gradients
    optimizer.zero_grad()

    # forward passing on all training flowers :3
    training_outputs = model(training_inputs_tensor)

    # checking how wrong are the predicted values to real ones
    loss = loss_function(training_outputs, training_targets_tensor)

    # calculate the gradient of the loss with respect to every weight in the network
    loss.backward()

    # update the weights a bit to make the loss smaller
    optimizer.step()

    if epoch_number % 20 == 0:
        pred_train_classes = torch.argmax(training_outputs, dim = 1)
        correct_preds = (pred_train_classes == training_targets_tensor).sum().item()
        acc_train = 100.0 * correct_preds / len(training_targets_tensor)
        print(f'epoch {epoch_number}/200, ')
        print(f'loss: {loss.item():.4f}, ')
        print(f'training accuracy: {acc_train:.2f}%')
        print('---------------')

epoch 20/200, 
loss: 0.5743, 
training accuracy: 86.67%
---------------
epoch 40/200, 
loss: 0.2920, 
training accuracy: 86.67%
---------------
epoch 60/200, 
loss: 0.2090, 
training accuracy: 93.33%
---------------
epoch 80/200, 
loss: 0.1485, 
training accuracy: 95.83%
---------------
epoch 100/200, 
loss: 0.1078, 
training accuracy: 95.00%
---------------
epoch 120/200, 
loss: 0.0877, 
training accuracy: 97.50%
---------------
epoch 140/200, 
loss: 0.0758, 
training accuracy: 97.50%
---------------
epoch 160/200, 
loss: 0.0680, 
training accuracy: 97.50%
---------------
epoch 180/200, 
loss: 0.0627, 
training accuracy: 98.33%
---------------
epoch 200/200, 
loss: 0.0587, 
training accuracy: 98.33%
---------------


In [17]:
model.eval()

with torch.no_grad():

    final_train_out = model(training_inputs_tensor)
    final_pred_train = torch.argmax(final_train_out, dim = 1)
    final_train_correct = (final_pred_train == training_targets_tensor).sum().item()
    final_train_acc = 100.0 * final_train_correct / len(training_targets_tensor)

    test_out = model(testing_inputs_tensor)
    test_loss = loss_function(test_out, testing_targets_tensor)

    pred_test = torch.argmax(test_out, dim=1)
    corr_pred_num = (pred_test == testing_targets_tensor).sum().item()
    test_acc = 100.0 * corr_pred_num / len(testing_targets_tensor)

print(f"final training accuracy: {final_train_acc:.2f}%")
print(f"final test accuracy: {test_acc:.2f}%")
print(f"final test loss: {test_loss.item():.4f}")

final training accuracy: 98.33%
final test accuracy: 100.00%
final test loss: 0.0445


hurray